# DiagnoTech Model 2 · Notebook 01 — Dataset Preparation & Analysis

**Purpose.** Inspect the supplied archive, verify labels and images, build a reproducible manifest, and create a leakage-free patient-level split (70 / 15 / 15).

**Read this first: the label format is assumed, then verified here.** The file name `NIH_Model2_7_Diseases.zip` and the eight requested classes (seven diseases plus *No Finding*) match the NIH ChestX-ray14 label vocabulary. In that dataset one image can carry several findings, for example `Finding Labels = "Effusion|Mass"`. The project therefore uses a **multi-label** formulation: eight independent sigmoid outputs trained with binary cross-entropy. A softmax single-label model would silently discard co-occurring findings. Section 4 checks this against the actual annotation file.

**Outputs** (written to Google Drive): `data/manifest_full.csv`, `data/manifest_{train,val,test}.csv`, `data/dataset_audit.json`, `data/split_summary.json`, `data/class_statistics.json`, `data/dataset_metadata.json`, and `reports/eda_*.png`.

## 1. Environment setup
Install the one dependency Colab does not provide by default, then locate the project code. Unzip `DiagnoTech_Model2_Professional.zip` so that `/content/DiagnoTech_Model2/src` exists (see the README).

In [ ]:
%pip install -q gdown

In [ ]:
import os
import platform
import shutil
import sys
from pathlib import Path

CANDIDATE_CODE_ROOTS = [
    os.environ.get("DIAGNOTECH_CODE_ROOT"),
    "/content/DiagnoTech_Model2",
    str(Path.cwd()),
    str(Path.cwd().parent),
]
code_roots = [Path(c) for c in CANDIDATE_CODE_ROOTS if c and (Path(c) / "src" / "diagnotech").is_dir()]
if not code_roots:
    raise FileNotFoundError(
        "Project code not found. Unzip DiagnoTech_Model2_Professional.zip so that /content/DiagnoTech_Model2/src exists."
    )
CODE_ROOT = code_roots[0]
sys.path.insert(0, str(CODE_ROOT / "src"))

from diagnotech.config import in_colab, load_context, save_json  # noqa: E402

if in_colab():
    from google.colab import drive
    drive.mount("/content/drive")  # first run asks for authorisation

ctx = load_context(CODE_ROOT)
cfg = ctx.project
CLASSES = list(ctx.classes)
SEED = cfg["split"]["seed"]
SPLIT_FRACTIONS = {"train": cfg["split"]["train"], "val": cfg["split"]["val"], "test": cfg["split"]["test"]}
ann_cfg = cfg["annotation"]
DATA = ctx.storage.data
REPORTS = ctx.storage.reports
print("Code root   :", CODE_ROOT)
print("Storage root:", ctx.storage.root)

In [ ]:
import pandas as pd
import sklearn
import torch

print(f"Python {platform.python_version()} | torch {torch.__version__} | pandas {pd.__version__} | scikit-learn {sklearn.__version__}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} | {props.total_memory / 1024**3:.1f} GB VRAM")
else:
    print("WARNING: no CUDA GPU. Preprocessing still works. Training in notebook 02 will be very slow without a GPU.")

RAW_DIR = Path(os.environ.get("DIAGNOTECH_RAW_DIR", cfg["source"]["colab_raw_data_dir"]))
RAW_DIR.parent.mkdir(parents=True, exist_ok=True)
free_gb = shutil.disk_usage(RAW_DIR.parent).free / 1024**3
print(f"Free disk for raw data: {free_gb:.1f} GB")
assert free_gb > 8, "About 8 GB free is needed for the archive and its extraction."

## 2. Acquire and extract the archive
The archive is about 2.4 GB. It is downloaded once per Colab session to local disk (`/content`), which reads far faster than Google Drive. Only the small manifests are written back to Drive. If you already have a mounted copy of the zip, set `LOCAL_ZIP_OVERRIDE`.

In [ ]:
import zipfile

ARCHIVE_PATH = RAW_DIR.parent / "source_archive.zip"
LOCAL_ZIP_OVERRIDE = None  # e.g. "/content/drive/MyDrive/<folder>/NIH_Model2_7_Diseases.zip"
FILE_ID = cfg["source"]["drive_file_id"]


def ensure_archive() -> Path:
    if LOCAL_ZIP_OVERRIDE:
        return Path(LOCAL_ZIP_OVERRIDE)
    if ARCHIVE_PATH.exists() and zipfile.is_zipfile(ARCHIVE_PATH):
        return ARCHIVE_PATH
    import gdown
    gdown.download(id=FILE_ID, output=str(ARCHIVE_PATH), quiet=False)
    if not (ARCHIVE_PATH.exists() and zipfile.is_zipfile(ARCHIVE_PATH)):
        raise RuntimeError(
            "The download did not produce a valid zip. Check that the Drive file is shared as "
            "'Anyone with the link', or set LOCAL_ZIP_OVERRIDE to a mounted copy."
        )
    return ARCHIVE_PATH


archive = ensure_archive()
print(f"Archive: {archive} ({archive.stat().st_size / 1024**3:.2f} GB)")

marker = RAW_DIR / ".extracted_ok"
if not marker.exists():
    with zipfile.ZipFile(archive) as zf:
        members = [m for m in zf.infolist() if not m.filename.startswith("__MACOSX")]
        for i, member in enumerate(members, start=1):
            zf.extract(member, RAW_DIR)
            if i % 10000 == 0:
                print(f"  extracted {i:,} / {len(members):,}")
    marker.write_text(archive.name)
print("Extracted to:", RAW_DIR)

## 3. Inspect the archive structure
Locate the annotation CSV and confirm it has the required columns. If the columns differ from the NIH layout, the notebook stops here rather than guessing.

In [ ]:
csv_files = sorted(RAW_DIR.rglob("*.csv"))
print(f"CSV files found: {len(csv_files)}")
for f in csv_files[:20]:
    print("  ", f.relative_to(RAW_DIR))
if not csv_files:
    raise FileNotFoundError("The archive contains no CSV annotation file.")

ANNOTATION_CSV = next((f for f in csv_files if f.name.lower().startswith("data_entry")), csv_files[0])
annotations = pd.read_csv(ANNOTATION_CSV)
required_cols = [ann_cfg["image_id_column"], ann_cfg["label_column"], ann_cfg["patient_id_column"]]
missing_cols = [c for c in required_cols if c not in annotations.columns]
print(f"Annotation file: {ANNOTATION_CSV.name} | rows: {len(annotations):,}")
print("Columns:", list(annotations.columns))
if missing_cols:
    raise ValueError(f"Required columns missing: {missing_cols}. Stop and report: the label format differs from the expected layout.")
annotations.head(3)

In [ ]:
from diagnotech.data_manifest import index_image_files

image_index, duplicate_names = index_image_files(RAW_DIR)
print(f"Image files indexed: {len(image_index):,} | duplicate file names skipped: {len(duplicate_names):,}")
if duplicate_names:
    print("  first duplicate names:", duplicate_names[:10])

## 4. Label verification: multi-label structure and all eight classes
Every label token is counted. The notebook then checks that each required class occurs in the annotations and measures how often images carry more than one finding. That second number decides whether the multi-label formulation is needed.

In [ ]:
from collections import Counter

token_counter: Counter = Counter()
multi_label_rows = 0
for raw in annotations[ann_cfg["label_column"]].dropna():
    tokens = [t.strip() for t in str(raw).split(ann_cfg["label_separator"]) if t.strip()]
    token_counter.update(tokens)
    if len(set(tokens)) > 1:
        multi_label_rows += 1

token_table = pd.DataFrame(token_counter.most_common(), columns=["label_token", "count"])
token_table["in_target_classes"] = token_table["label_token"].isin(CLASSES)
print(f"Rows with more than one distinct label: {multi_label_rows:,} of {len(annotations):,} "
      f"({100 * multi_label_rows / max(len(annotations), 1):.1f}%)")
display(token_table)

absent = [c for c in CLASSES if token_counter.get(c, 0) == 0]
if absent:
    raise ValueError(f"These target classes never occur in the annotations: {absent}. Do not proceed silently.")
print("All eight target classes occur in the annotations.")

## 5. Manifest construction and image validation
Each annotation row is joined to its image file. Every image is then opened to detect corruption. Each file is hashed to detect exact duplicates. Duplicates matter for leakage: the same picture under two patient IDs would otherwise land in both training and test data.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

from diagnotech.data_manifest import build_manifest, file_sha1, inspect_image, resolve_image_paths

manifest_raw, audit = build_manifest(annotations, image_index, ann_cfg, CLASSES)
print(f"Manifest rows before image validation: {len(manifest_raw):,}")
print("Label statuses:", audit["label_status_counts"])

with ThreadPoolExecutor(max_workers=8) as pool:
    checks = list(pool.map(inspect_image, resolve_image_paths(manifest_raw, RAW_DIR)["image_path"]))
check_table = pd.DataFrame([c.__dict__ for c in checks])
ok_mask = check_table["ok"].to_numpy(dtype=bool)
audit["corrupt_images"] = int((~ok_mask).sum())
print(f"Corrupted or unreadable images: {audit['corrupt_images']:,}")
if (~ok_mask).any():
    print(check_table.loc[~ok_mask, ["path", "error"]].head(10).to_string())

manifest_valid = manifest_raw.loc[ok_mask].reset_index(drop=True)
print("Most common image geometries (width x height, mode):")
print(check_table.loc[ok_mask].groupby(["width", "height", "mode"]).size().sort_values(ascending=False).head(8).to_string())

In [ ]:
with ThreadPoolExecutor(max_workers=8) as pool:
    manifest_valid["sha1"] = list(pool.map(file_sha1, resolve_image_paths(manifest_valid, RAW_DIR)["image_path"]))

hash_counts = manifest_valid.groupby("sha1").size()
cross_patient = manifest_valid.groupby("sha1")["patient_id"].nunique()
audit["duplicate_hash_groups"] = int((hash_counts > 1).sum())
audit["images_in_duplicate_groups"] = int(hash_counts[hash_counts > 1].sum())
audit["cross_patient_duplicate_groups"] = int((cross_patient > 1).sum())
print(f"Exact-duplicate groups: {audit['duplicate_hash_groups']:,} | "
      f"groups spanning several patients: {audit['cross_patient_duplicate_groups']:,}")

## 6. Patient-level 70 / 15 / 15 split with leakage checks
Patients are the unit of separation, so all images from one patient stay in one split. Patients who share an identical image file are merged into one group. Groups containing rare findings are placed first, so rare classes spread across the splits instead of collecting in one. The split is seeded and deterministic. `verify_split` raises an error if any leakage remains.

In [ ]:
from diagnotech.data_manifest import build_leakage_groups, patient_level_split, save_manifest, verify_split

manifest_valid["group_id"] = build_leakage_groups(manifest_valid, "patient_id", hash_col="sha1")
manifest_valid["split"] = patient_level_split(manifest_valid, SPLIT_FRACTIONS, SEED, "group_id", CLASSES)
split_summary = verify_split(
    manifest_valid, "split", "group_id", CLASSES,
    min_positives=cfg["split"]["min_positives_per_class_eval_split"],
)
print("Rows per split    :", split_summary["rows_per_split"])
print("Patients per split:", split_summary["patients_per_split"])
for warning in split_summary["warnings"]:
    print("WARNING:", warning)

positives_table = pd.DataFrame(split_summary["positives_per_split"]).T[CLASSES]
display(positives_table)

save_manifest(manifest_valid, DATA / "manifest_full.csv")
for split in ("train", "val", "test"):
    save_manifest(manifest_valid.loc[manifest_valid["split"] == split].reset_index(drop=True), DATA / f"manifest_{split}.csv")
save_json(audit, DATA / "dataset_audit.json")
save_json(split_summary, DATA / "split_summary.json")
print("Manifests written to", DATA)

## 7. Class statistics from the training split only
Class weights come only from training data. The validation and test splits never influence them. The weight used in the loss is `negatives / positives` per class, which is the inverse-frequency weight for independent (multi-label) binary cross-entropy.

In [ ]:
from diagnotech.data_manifest import class_statistics

train_df = manifest_valid.loc[manifest_valid["split"] == "train"]
stats = class_statistics(train_df, CLASSES, pos_weight_max=cfg["training"]["pos_weight_max"])
save_json(stats, DATA / "class_statistics.json")
display(pd.DataFrame({
    "train_positives": stats["positives"],
    "prevalence": stats["prevalence"],
    "pos_weight": stats["pos_weight"],
}))

## 8. Exploratory analysis
Class prevalence by split, label cardinality, the label co-occurrence matrix, and sample images for every class.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

plt.rcParams.update({"figure.dpi": 110})

prevalence_by_split = manifest_valid.groupby("split")[CLASSES].sum()
fig, ax = plt.subplots(figsize=(11, 4.8))
prevalence_by_split.T.loc[CLASSES].plot(kind="bar", stacked=True, ax=ax)
ax.set_title("Positive labels per class, stacked by split")
ax.set_ylabel("number of positive labels")
ax.set_xlabel("")
ax.legend(title="split")
fig.tight_layout()
fig.savefig(REPORTS / "eda_class_distribution.png", bbox_inches="tight")
plt.show()

label_matrix = manifest_valid[CLASSES].to_numpy(dtype=np.int64)
cooccurrence = label_matrix.T @ label_matrix
fig, axes = plt.subplots(1, 2, figsize=(15, 5.8))
axes[0].hist(manifest_valid["num_positive_labels"], bins=np.arange(0.5, len(CLASSES) + 1.5, 1), rwidth=0.8)
axes[0].set_title("Findings per image (label cardinality)")
axes[0].set_xlabel("number of positive labels")
axes[0].set_ylabel("images")

im = axes[1].imshow(cooccurrence, cmap="Blues")
axes[1].set_xticks(range(len(CLASSES)))
axes[1].set_xticklabels(CLASSES, rotation=45, ha="right")
axes[1].set_yticks(range(len(CLASSES)))
axes[1].set_yticklabels(CLASSES)
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        axes[1].text(j, i, f"{cooccurrence[i, j]:,}", ha="center", va="center", fontsize=7)
axes[1].set_title("Label co-occurrence (images)")
fig.tight_layout()
fig.savefig(REPORTS / "eda_cardinality_cooccurrence.png", bbox_inches="tight")
plt.show()

rng = np.random.default_rng(SEED)
fig, axes = plt.subplots(len(CLASSES), 3, figsize=(9, 3.0 * len(CLASSES)))
for row_idx, cls in enumerate(CLASSES):
    pool_rows = train_df[train_df[cls] == 1]
    picks = pool_rows.sample(n=min(3, len(pool_rows)), random_state=SEED)
    for col_idx in range(3):
        ax = axes[row_idx, col_idx]
        ax.axis("off")
        if col_idx < len(picks):
            record = picks.iloc[col_idx]
            with Image.open(RAW_DIR / record["relative_path"]) as img:
                ax.imshow(img.convert("L"), cmap="gray")
            ax.set_title(f"{cls}\nlabels: {record['finding_labels']}", fontsize=7)
fig.tight_layout()
fig.savefig(REPORTS / "eda_sample_images.png", bbox_inches="tight")
plt.show()

## 9. Dataset metadata and validation checks
A last set of assertions protects downstream notebooks. If any check fails, do not start training.

In [ ]:
metadata = {
    "source_drive_file_id": FILE_ID,
    "source_file_name": cfg["source"]["drive_file_name"],
    "archive_bytes": int(archive.stat().st_size),
    "annotation_file": ANNOTATION_CSV.name,
    "annotation_columns": required_cols,
    "classes": CLASSES,
    "label_type": "multi-label (independent sigmoid per class)",
    "split_fractions": SPLIT_FRACTIONS,
    "seed": SEED,
    "image_size": cfg["preprocessing"]["image_size"],
    "normalization": {"mean": cfg["preprocessing"]["mean"], "std": cfg["preprocessing"]["std"]},
    "python": platform.python_version(),
    "torch": torch.__version__,
    "rows_final": int(len(manifest_valid)),
    "patients_final": int(manifest_valid["patient_id"].nunique()),
}
save_json(metadata, DATA / "dataset_metadata.json")

assert set(manifest_valid["split"].unique()) == {"train", "val", "test"}, "A split is empty"
assert manifest_valid[CLASSES].sum(axis=1).min() >= 1, "Found a row with no positive label"
assert not manifest_valid.groupby("group_id")["split"].nunique().gt(1).any(), "Leakage: a group spans splits"
assert not manifest_valid.groupby("patient_id")["split"].nunique().gt(1).any(), "Leakage: a patient spans splits"
print("All dataset validation checks passed.")

## Next step
Run **02_DenseNet121_Training.ipynb** in the same Colab session. It reads only the `train` and `val` manifests. The test split is not loaded until notebook 03.